# Teleco data explained 

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("../data/raw_data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [3]:
data.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [7]:
metadata = {
    'customerID': 'customer ID',
    'gender' : 'customer gender',
    'SeniorCitizen': 'Whether the customer is a senior citizen or not (1, 0)',
    'Partner': 'Whether the customer has a partner or not (Yes, No)',
    'Dependents' : 'Whether the customer has dependents or not (Yes, No)',
    'tenure' : 'Number of months the customer has stayed with the company',
    'PhoneService': 'Whether the customer has a phone service or not (Yes, No)',
    'MultipleLines' : 'Whether the customer has multiple lines or not (Yes, No, No phone service)',
    'InternetService' : 'Customer’s internet service provider (DSL, Fiber optic, No)',
    'OnlineSecurity' : 'Whether the customer has online security or not (Yes, No, No internet service)',
    'OnlineBackup' : 'Whether the customer has online backup or not (Yes, No, No internet service)',
    'DeviceProtection' : 'Whether the customer has device protection or not (Yes, No, No internet service)',
    'TechSupport' : 'Whether the customer has tech support or not (Yes, No, No internet service)',
    'StreamingTV' : 'Whether the customer has streaming TV or not (Yes, No, No internet service)',
    'StreamingMovies' : 'Whether the customer has streaming movies or not (Yes, No, No internet service)',
    'Contract' : 'The contract term of the customer (Month-to-month, One year, Two year)',
    'PaperlessBilling' : 'Whether the customer has paperless billing or not (Yes, No)',
    'PaymentMethod' : 'The customer’s payment method (Electronic check, Mailed check, Bank transfer (automatic), Credit card (automatic))',
    'MonthlyCharges' : 'The amount charged to the customer monthly',
    'TotalCharges' : 'The total amount charged to the customer',
    'Churn' : 'Whether the customer churned or not (Yes or No)'
}

In [20]:
metadataset = pd.DataFrame( columns=['Column' , 'Description'])

In [29]:
counter = 0
for key , value in metadata.items():
    metadataset.loc[counter,"Column"] = key
    metadataset.loc[counter,"Description"] = value
    counter +=1

In [30]:
metadataset

,Column,Description
0,customerID,customer ID
1,gender,customer gender
2,SeniorCitizen,Whether the customer is a senior citizen or no...
3,Partner,Whether the customer has a partner or not (Yes...
4,Dependents,Whether the customer has dependents or not (Ye...
5,tenure,Number of months the customer has stayed with ...
6,PhoneService,Whether the customer has a phone service or no...
7,MultipleLines,Whether the customer has multiple lines or not...
8,InternetService,"Customer’s internet service provider (DSL, Fib..."
9,OnlineSecurity,Whether the customer has online security or no...


In [32]:
# saving data
metadataset.to_csv('../data/raw_data/metadata.csv')

---

# 📊 Telco Churn — Data Understanding & EDA Guide

> **Business Objective:** Predict which customers are likely to churn so the business can intervene proactively with targeted retention programs — reducing revenue loss and improving customer lifetime value (CLV).

---

## 1. Business Context

Customer churn is one of the most expensive problems in the telecom industry. Acquiring a new customer costs **5–25× more** than retaining an existing one. Every churned customer represents:

- Lost recurring monthly revenue
- Wasted acquisition cost (CAC)
- A potential brand detractor

The dataset captures a **snapshot in time** — customers who **have already churned** in the last month (`Churn = Yes`) vs. those who stayed (`Churn = No`). Your model must learn behavioral and demographic signals that **preceded** that churn decision.

---

## 2. Dataset Overview

| Property | Detail |
|---|---|
| **Unit of observation** | One row = one unique customer |
| **Target variable** | `Churn` (Yes / No) — binary classification |
| **Total features** | 20 input columns + 1 target |
| **Source** | IBM Sample Data Sets |
| **Expected class imbalance** | ~26–27% churn rate (minority class) — **handle carefully** |

---

## 3. Column Inventory & Business Meaning

### 🔑 Identifier
| Column | Type | Values | EDA Notes |
|---|---|---|---|
| `customerID` | String/ID | Unique per row | Drop before modeling. Check for duplicates first. |

---

### 👤 Demographics
| Column | Type | Values | Business Meaning | EDA Priority |
|---|---|---|---|---|
| `gender` | Binary categorical | Male, Female | Gender of the customer | Low — typically weak predictor in telecom |
| `SeniorCitizen` | Binary numeric | 0, 1 | Whether customer is 65+ | **High** — seniors may have different service needs and churn differently |
| `Partner` | Binary categorical | Yes, No | Has a spouse/domestic partner | Medium — partnered customers tend to be more stable |
| `Dependents` | Binary categorical | Yes, No | Has children or other dependents | Medium — families tend to have lower churn; linked to contract type |

> 💡 **Business insight:** Customers with partners AND dependents form a "family unit" — they are often more price-sensitive but also more loyal. Consider building a `is_family` engineered feature.

---

### 🕐 Tenure
| Column | Type | Values | Business Meaning | EDA Priority |
|---|---|---|---|---|
| `tenure` | Continuous numeric | 0–72 months | How long the customer has been with the company | **Critical** — the single strongest predictor of churn; new customers churn far more |

> 💡 **Business insight:** Churn risk is **highest in the first 12 months**. Look for a hockey-stick drop in churn rate as tenure increases. Consider binning into `Early (0–12)`, `Mid (13–36)`, `Loyal (37+)`.

---

### 📞 Phone Services
| Column | Type | Values | Business Meaning | EDA Priority |
|---|---|---|---|---|
| `PhoneService` | Binary categorical | Yes, No | Has a basic phone line | Low — ~90% of customers have it; low variance |
| `MultipleLines` | Ternary categorical | Yes, No, No phone service | Has more than one line | Medium — `No phone service` is a **subset** of PhoneService=No, handle as hierarchy |

> ⚠️ **EDA Watch:** `MultipleLines = "No phone service"` is structurally tied to `PhoneService = "No"`. These are **not independent**. Don't treat them as separate binary flags without acknowledging this dependency.

---

### 🌐 Internet & Add-on Services
| Column | Type | Values | Business Meaning | EDA Priority |
|---|---|---|---|---|
| `InternetService` | Ternary categorical | DSL, Fiber optic, No | Type of internet subscription | **Critical** — Fiber optic customers tend to churn most; high cost may drive dissatisfaction |
| `OnlineSecurity` | Ternary categorical | Yes, No, No internet service | Security add-on | High — lack of security may correlate with dissatisfaction |
| `OnlineBackup` | Ternary categorical | Yes, No, No internet service | Backup add-on | Medium |
| `DeviceProtection` | Ternary categorical | Yes, No, No internet service | Device protection plan | Medium |
| `TechSupport` | Ternary categorical | Yes, No, No internet service | Tech support add-on | **High** — no support = frustration = churn |
| `StreamingTV` | Ternary categorical | Yes, No, No internet service | Streaming TV subscription | Medium — engagement feature; more services = more "locked in" |
| `StreamingMovies` | Ternary categorical | Yes, No, No internet service | Streaming movies subscription | Medium |

> ⚠️ **EDA Watch:** All 6 add-on columns share the same **"No internet service"** category. This creates a **hierarchical dependency** on `InternetService`. Before encoding these, you must decide: collapse `"No internet service"` into `"No"` or keep as a 3-level ordinal? Your choice **will affect model behavior**.

> 💡 **Engineered feature idea:** Create a `num_addons` count (0–6) from the 6 add-on services. More add-ons = more product stickiness = lower expected churn. Test this hypothesis in EDA.

---

### 💳 Account & Billing
| Column | Type | Values | Business Meaning | EDA Priority |
|---|---|---|---|---|
| `Contract` | Ordinal categorical | Month-to-month, One year, Two year | Contract commitment level | **Critical** — Month-to-month customers have zero switching cost; highest churn risk |
| `PaperlessBilling` | Binary categorical | Yes, No | Opted into paperless billing | Medium — proxy for tech-savviness; may correlate with fiber optic usage |
| `PaymentMethod` | Nominal categorical | Electronic check, Mailed check, Bank transfer (automatic), Credit card (automatic) | How the customer pays | **High** — Electronic check is associated with higher churn; automatic payment = more friction to leave |
| `MonthlyCharges` | Continuous numeric | ~$18–$119 | Current monthly bill | **Critical** — Higher charges = higher churn probability; check distribution |
| `TotalCharges` | Continuous numeric | ~$18–$8,700 | Cumulative billing to date | High — strongly correlated with `tenure × MonthlyCharges`; check for multicollinearity |

> ⚠️ **EDA Watch:** `TotalCharges` is **not independent** of `tenure` and `MonthlyCharges`. Expect near-perfect multicollinearity: `TotalCharges ≈ tenure × MonthlyCharges`. You likely cannot use all three in a linear model without VIF analysis.

> ⚠️ **Data Quality Alert:** `TotalCharges` is sometimes stored as a string/object type in this dataset due to whitespace in empty cells. **Always check `df['TotalCharges'].dtype` and coerce to numeric with `pd.to_numeric(..., errors='coerce')`**. Missing values created this way are not random — they correspond to `tenure = 0` (brand-new customers).

---

### 🎯 Target Variable
| Column | Type | Values | Notes |
|---|---|---|---|
| `Churn` | Binary categorical | Yes, No | Convert to 1/0 for modeling. Expected ~26% positive class. |

---

## 4. Feature Groups Summary

```
┌─────────────────────────────────────────────────────────┐
│  DEMOGRAPHICS      │  gender, SeniorCitizen,            │
│                    │  Partner, Dependents               │
├────────────────────┼────────────────────────────────────┤
│  TENURE            │  tenure                            │
├────────────────────┼────────────────────────────────────┤
│  PHONE SERVICES    │  PhoneService, MultipleLines       │
├────────────────────┼────────────────────────────────────┤
│  INTERNET & ADD-ONS│  InternetService, OnlineSecurity,  │
│                    │  OnlineBackup, DeviceProtection,   │
│                    │  TechSupport, StreamingTV,         │
│                    │  StreamingMovies                   │
├────────────────────┼────────────────────────────────────┤
│  ACCOUNT/BILLING   │  Contract, PaperlessBilling,       │
│                    │  PaymentMethod, MonthlyCharges,    │
│                    │  TotalCharges                      │
├────────────────────┼────────────────────────────────────┤
│  TARGET            │  Churn                             │
└─────────────────────────────────────────────────────────┘
```

---

## 5. EDA Checklist — Step by Step

### ✅ Step 1 — Sanity Checks (Before Anything Else)
- [ ] `df.shape` — confirm row/column count
- [ ] `df.dtypes` — identify mistyped columns (especially `TotalCharges`)
- [ ] `df.duplicated().sum()` — check for duplicate `customerID`
- [ ] `df.isnull().sum()` — surface missing values
- [ ] `df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')` — fix the known type issue
- [ ] `df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})` — encode target

---

### ✅ Step 2 — Target Variable Analysis
- [ ] Plot `Churn` value counts and class ratio — **expect ~26% churn**
- [ ] Document the imbalance ratio → this dictates whether you need SMOTE / class weights in modeling
- [ ] This is your **reference distribution** — every subsequent plot should be split by Churn

---

### ✅ Step 3 — Univariate Analysis

**For continuous features** (`tenure`, `MonthlyCharges`, `TotalCharges`):
- [ ] Histogram + KDE — check for skewness, bimodality
- [ ] Box plot — spot outliers
- [ ] Descriptive stats: mean, median, std, min, max, quartiles
- [ ] `tenure`: expect a **bimodal distribution** — many new customers + many long-tenured loyalists
- [ ] `MonthlyCharges`: expect **right skew** driven by premium fiber optic plans

**For categorical features** (all others):
- [ ] Bar chart of value counts for each column
- [ ] Note class imbalances within features (e.g., only ~16% are SeniorCitizen)

---

### ✅ Step 4 — Bivariate Analysis (vs. Churn) ⭐ Most Important Step

**Continuous vs. Churn:**
- [ ] `tenure` by Churn — box plot or violin plot; **expect median tenure to be much lower for churners**
- [ ] `MonthlyCharges` by Churn — churners likely pay more
- [ ] `TotalCharges` by Churn — churners likely have lower total (short tenure)
- [ ] KDE plots: overlay `Churn=0` and `Churn=1` distributions for all three

**Categorical vs. Churn:**
- [ ] Stacked/grouped bar chart showing **churn rate per category** for every categorical column
- [ ] Key hypotheses to validate:
  - `Contract = Month-to-month` → highest churn rate
  - `InternetService = Fiber optic` → higher churn than DSL
  - `PaymentMethod = Electronic check` → highest churn
  - `TechSupport = No` → higher churn
  - `SeniorCitizen = 1` → higher churn
  - `tenure < 12` → highest churn risk group

---

### ✅ Step 5 — Multivariate Analysis

- [ ] **Correlation heatmap** — numeric features only (`tenure`, `MonthlyCharges`, `TotalCharges`)
  - Confirm near-perfect correlation between `TotalCharges` and `tenure`
  - Compute VIF if planning a linear/logistic model
- [ ] **Pairplot** — `tenure`, `MonthlyCharges`, `TotalCharges` colored by Churn
- [ ] **Contract × InternetService × Churn** — 2D heatmap of churn rate; fiber + month-to-month is your highest-risk segment
- [ ] **tenure × Contract × Churn** — how does early tenure interact with contract type?

---

### ✅ Step 6 — Service Bundling Analysis

- [ ] Compute `num_addons` = count of Yes across 6 internet add-on columns
- [ ] Plot churn rate by `num_addons` — expect monotonically decreasing churn as add-ons increase
- [ ] Identify customers with `InternetService = No` and `PhoneService = No` — are these churners or non-churners?

---

### ✅ Step 7 — Data Quality & Special Cases

- [ ] Rows where `tenure = 0` — are these data errors or true new customers? Cross-check with `TotalCharges`
- [ ] Validate `MultipleLines` / add-on "No internet service" rows align perfectly with `InternetService = No`
- [ ] Check if any customer with `PhoneService = No` has `MultipleLines ≠ "No phone service"` (should not exist)

---

## 6. Key Hypotheses to Test in EDA

| # | Hypothesis | Expected Finding |
|---|---|---|
| H1 | Month-to-month contract customers churn at a higher rate | ✅ Likely True — no lock-in |
| H2 | Fiber optic internet customers churn more than DSL | ✅ Likely True — higher cost, higher expectations |
| H3 | Customers with lower tenure churn more | ✅ Likely True — early dissatisfaction |
| H4 | Higher monthly charges increase churn probability | ✅ Likely True — price sensitivity |
| H5 | Electronic check users churn more than automatic payment users | ✅ Likely True — lower friction to cancel |
| H6 | Senior citizens have higher churn rate | ✅ Likely True — service complexity friction |
| H7 | Customers with tech support churn less | ✅ Likely True — problem resolution reduces frustration |
| H8 | More add-on services = lower churn (stickiness effect) | ✅ Likely True — higher switching cost |
| H9 | TotalCharges is nearly redundant given tenure + MonthlyCharges | ✅ Likely True — multicollinearity concern |
| H10 | Customers without partners or dependents churn more | ⚠️ Test — solo customers may be more mobile |

---

## 7. Known Data Pitfalls (Don't Miss These)

| Pitfall | Description | Fix |
|---|---|---|
| `TotalCharges` as string | Whitespace causes object dtype | `pd.to_numeric(..., errors='coerce')` |
| Hierarchical categoricals | 6 add-on columns nest under `InternetService` | Decide encoding strategy upfront |
| Multicollinearity | `TotalCharges ≈ tenure × MonthlyCharges` | Drop one or use PCA/VIF |
| Class imbalance | ~26% churn minority | Use stratified splits, SMOTE, or `class_weight='balanced'` |
| `SeniorCitizen` as int | Already 0/1, not "Yes/No" — inconsistent with others | Document; don't re-encode accidentally |

---

## 8. Suggested Feature Engineering (Post-EDA)

| Feature | Formula / Logic | Rationale |
|---|---|---|
| `num_addons` | Count of "Yes" in 6 add-on columns | Measures product stickiness |
| `is_family` | `Partner == Yes AND Dependents == Yes` | Family stability indicator |
| `is_auto_pay` | `PaymentMethod` in [Bank transfer, Credit card] | Proxy for payment commitment |
| `charges_per_month_ratio` | `TotalCharges / tenure` (guard ÷0) | Detect billing anomalies |
| `tenure_band` | Bin tenure into Early/Mid/Loyal | Non-linear tenure effect |
| `high_value_at_risk` | `tenure < 12 AND MonthlyCharges > median` | Priority retention target |

---

## 9. EDA Output Deliverables

By the end of EDA, you should be able to answer:

1. **What does the churn rate look like overall, and is it balanced?**
2. **Which single feature is most predictive of churn?**
3. **Which customer segment has the highest churn risk?** (your retention program's primary target)
4. **Are there any data quality issues that must be resolved before modeling?**
5. **Which features are redundant or collinear?**
6. **What feature engineering makes business sense?**

---

*Document prepared for Telco Churn EDA — IBM Sample Dataset*  
*Treat every visualization as a business story, not just a chart.*